In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Titanic-Dataset.csv')

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])


# Переводим категориальные признаки в числа (One-Hot Encoding)
df_ml = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Отбираем признаки для обучения и целевую переменную
X = df_ml[['Pclass', 'Age', 'Fare', 'FamilySize', 'Sex_male', 'Embarked_Q', 'Embarked_S']]
y = df_ml['Survived']

X = X.fillna(0)

# Переводим категориальные признаки в числа (One-Hot Encoding)

# Разделяем на обучающую и тестовую выборки (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Масштабирование признаков (критично для линейных моделей и KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

# Инициализируем словари для моделей и их результатов
models = {
    "Linear (Logistic Regression)": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, random_state=42, eval_metric='logloss'),
    "CatBoost": CatBoostClassifier(iterations=150, depth=4, random_state=42, verbose=0)
}

results = {}


# Обучение и оценка
for name, model in models.items():
    # Для линейной регрессии и KNN используем отмасштабированные данные
    if name in ["Linear (Logistic Regression)", "KNN"]:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    
    # Считаем метрику Accuracy (точность)
    results[name] = accuracy_score(y_test, preds)

# Сортировка результатов по убыванию точности
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
df_results = pd.DataFrame(sorted_results, columns=['Модель', 'Accuracy (Точность)'])
df_results['Accuracy (Точность)'] = df_results['Accuracy (Точность)'].map('{:.2%}'.format)
df_results